# Skyline Correlations and an A/B Design

Apply the correlation and A/B testing  to the Skyline Online Courses dataset

Author: Meron Welderufael
Date: 08/19/2026

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px

In [2]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

grade_map = {
    "F": 0,
    "D": 1,
    "C": 2,
    "B": 3,
    "A": 4
}

skyline["final_grade"]=skyline["final_grade"].map(grade_map)

pearson_r, pearson_p = stats.pearsonr(
    skyline["hours_studied"],
    skyline["final_grade"]
)

spearman_r, spearman_p = stats.spearmanr(
    skyline["hours_studied"],
    skyline["final_grade"]
)

print(f"Pearson r: {pearson_r:.3f} , (p: {pearson_p:.4f})")
print(f"Spearman r: {spearman_r:.3f} , (p: {spearman_r:.4f})")




Pearson r: -0.029 , (p: 0.7724)
Spearman r: -0.047 , (p: -0.0467)


In [3]:
fig = px.scatter(
   skyline,
       x="hours_studied",
       y="final_grade",
       title = "",
       trendline="ols",
)

fig.show()

### Interpretation

The Pearson and Spearman correlations agree that there is a very weak negative relationship between hours studied and final grade. The scatter plot also shows a nearly flat trend line, suggesting that hours studied has little relationship with final grade in this dataset.

This relationship does not provide evidence that studying more hours causes a higher grade. To claim causation, we would need a controlled experiment, such as randomly assigning students to different amounts of study time and comparing their final grades. This would help control for other factors, such as prior knowledge, motivation, attendance, and study habits.

Possible Confounders

1. Prior academic ability: Students who already have stronger academic skills may both study more hours and earn higher final grades. This could create a positive correlation even if the additional studying itself did not cause the higher grades.

2. Motivation: Highly motivated students may be more likely to spend more time studying and also perform better on exams and assignments. In this case, motivation could influence both hours_studied and final_grade, creating a correlation without proving that studying caused the higher grade. 

Part C: A/B Test Design — Study Planner Feature

1. The treatment and control conditions
- Treatment: Students are enrolled with the auto-scheduling study planner turned on for the term.
- Control: Students use the platform as normal, with no study planner

2. The randomization unit (per-student, per-enrollment, per-cohort?)
- Per-student, randomized at the start of the enrollment period. Not per-cohort, because cohort-level randomization would require far more clusters to reach adequate power (and risks confounding by cohort-specific effects like instructor or semester). Not per-enrollment either, since the same student could show up in both arms across multiple courses, causing contamination (they might apply planner habits learned in the treatment course to their control course).

3. The outcome metric
- Primary: Proportion of students earning a final grade of B or higher (binary outcome — easy to interpret and matches the business question of "did it help students succeed").

4. The minimum effect size that would be practically meaningful (your judgment; defend it briefly)
- A 5-percentage-point lift (40% → 45% earning B or higher) is chosen as the smallest effect worth caring about. Reasoning: rolling out a new feature has engineering and support costs, so a 1–2 point bump wouldn't justify company-wide investment, but a 5-point jump translates to meaningfully more students hitting grade thresholds (e.g., for scholarships or program requirements) — a business-relevant, detectable threshold.

5. The required sample size, computed from a power analysis at α=0.05 and 80% power

For the sample size calculation, you can assume the baseline final grade rate of "B or higher" is 40% in your generated data (compute the actual baseline if you want to be precise) and that you want to detect a lift to 45%

- input 
Baseline rate: 40%
Target rate: 45%
Significance level: α = 0.05
Statistical power: 80%
Two-sided test
n ≈ (z_α/2 + z_β)² * (p₁(1-p₁) + p₂(1-p₂)) / δ²
-~1,531 students per group (≈3,062 total), rounded up to ~1,550/group (3,100 total) for a safety margin against dropout/missing data.
